# Elastic Net regression

In [1]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet, ElasticNetCV, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV, SGDRegressor
from sklearn.model_selection import train_test_split
warnings.simplefilter("ignore")

## Diabetes with few training observations: four models, each tuned by cross-validation

In [2]:
X, y = load_diabetes(return_X_y=True)
print("correlation of s1 and s2:", np.corrcoef(X[:, 4], X[:, 5])[0, 1].round(3))
alphas = np.logspace(-4, 1, 40)

def compare(n_train, n_splits=40):
    """Mean test R2 of four models, each penalty tuned by 5-fold CV on the training part only."""
    scores = {"Linear regression": [], "RidgeCV": [], "LassoCV": [], "ElasticNetCV": []}
    zeros = {"LassoCV": [], "ElasticNetCV": []}     # features set to 0; s1, s2 set to 0
    for s in range(n_splits):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=n_train, random_state=s)
        models = {"Linear regression": LinearRegression(), "RidgeCV": RidgeCV(alphas=alphas),
                  "LassoCV": LassoCV(alphas=alphas, cv=5),
                  "ElasticNetCV": ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1], alphas=alphas, cv=5)}
        for name, m in models.items():
            scores[name].append(m.fit(X_tr, y_tr).score(X_te, y_te))
            if name in zeros:
                zeros[name].append(((m.coef_ == 0).sum(), (m.coef_[[4, 5]] == 0).sum()))
    for name, v in scores.items():
        print(f"{name:18s} mean test R2 {np.mean(v):.3f}")
    for name, v in zeros.items():
        z = np.mean(v, axis=0)
        print(f"{name:18s} features set to 0: {z[0]:.2f} of 10; of the pair s1, s2: {z[1]:.2f}")

print("80 training observations, 40 random splits")
compare(80)

correlation of s1 and s2: 0.897
80 training observations, 40 random splits


Linear regression  mean test R2 0.412
RidgeCV            mean test R2 0.440
LassoCV            mean test R2 0.431
ElasticNetCV       mean test R2 0.437
LassoCV            features set to 0: 3.30 of 10; of the pair s1, s2: 1.35
ElasticNetCV       features set to 0: 1.98 of 10; of the pair s1, s2: 0.75


## The same comparison with the full training set

In [3]:
print("All 353 training observations (test size 0.2), 40 random splits")
compare(353)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

All 353 training observations (test size 0.2), 40 random splits


Linear regression  mean test R2 0.455
RidgeCV            mean test R2 0.455
LassoCV            mean test R2 0.456
ElasticNetCV       mean test R2 0.457
LassoCV            features set to 0: 1.73 of 10; of the pair s1, s2: 0.78
ElasticNetCV       features set to 0: 1.05 of 10; of the pair s1, s2: 0.45


## The same model with SGDRegressor

In [4]:
sgd = SGDRegressor(penalty="elasticnet", alpha=0.001, l1_ratio=0.5, max_iter=2000, random_state=0)
sgd.fit(X_train, y_train)
print("SGDRegressor elasticnet test R2", round(sgd.score(X_test, y_test), 4))

SGDRegressor elasticnet test R2 0.424


## Correlated inputs: the grouping effect

In [5]:
rng = np.random.default_rng(0)
n = 200
z = rng.standard_normal(n)
X2 = np.c_[z + 0.05 * rng.standard_normal(n), z + 0.05 * rng.standard_normal(n), z + 0.05 * rng.standard_normal(n),
           rng.standard_normal((n, 3))]
y2 = 3 * z + rng.standard_normal(n)
for name, m in [("Linear regression", LinearRegression()), ("Ridge(alpha=10)", Ridge(alpha=10)),
                ("Lasso(alpha=0.1)", Lasso(alpha=0.1)), ("ElasticNet(alpha=0.1, l1_ratio=0.5)", ElasticNet(alpha=0.1, l1_ratio=0.5))]:
    print(f"{name:36s}", m.fit(X2, y2).coef_.round(2))

Linear regression                    [-1.01  1.79  2.28  0.04  0.07  0.03]
Ridge(alpha=10)                      [0.92 1.04 1.06 0.04 0.05 0.02]
Lasso(alpha=0.1)                     [0.   1.27 1.68 0.   0.   0.  ]
ElasticNet(alpha=0.1, l1_ratio=0.5)  [0.89 1.02 1.04 0.   0.   0.  ]
